# 🚀 Day 4: Quantum Circuits as Matrix Operations

## 14-Day Quantum DevRel Bootcamp

**Goal:** Master the **circuit-matrix correspondence** — every quantum circuit IS a unitary matrix, and every unitary matrix IS a quantum circuit.

**Today's Deliverables:**
1. ✅ Build multi-qubit gate matrices by hand and verify against Qiskit
2. ✅ SWAP gate from three CNOTs
3. ✅ Controlled gates: CZ, controlled-U, CZ from CNOT
4. ✅ Toffoli gate decomposition into 1- and 2-qubit gates
5. ✅ Circuit simulation with measurement statistics (Qiskit Aer)
6. ✅ Universal gate sets and why {H, T, CNOT} is enough

**Key insight:** Understanding gate matrices lets you verify circuits algebraically, optimize them, and understand what transpilers do under the hood.

---

## Section 1: The Circuit-Matrix Correspondence

In [ ]:
# Core imports
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector, DensityMatrix, partial_trace
from qiskit_aer import AerSimulator

print("✅ All imports loaded!")
print()

# ── Standard single-qubit matrices ──
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H_mat = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
S = np.array([[1, 0], [0, 1j]], dtype=complex)
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)

# Projectors
P0 = np.array([[1, 0], [0, 0]], dtype=complex)  # |0⟩⟨0|
P1 = np.array([[0, 0], [0, 1]], dtype=complex)  # |1⟩⟨1|

print("📐 The Circuit-Matrix Correspondence")
print("=" * 55)
print()
print("Every quantum circuit is a unitary matrix:")
print("  • 1 qubit  →  2×2 matrix   (4 complex numbers)")
print("  • 2 qubits →  4×4 matrix   (16 complex numbers)")
print("  • 3 qubits →  8×8 matrix   (64 complex numbers)")
print("  • n qubits →  2ⁿ×2ⁿ matrix (exponential!)")
print()
print("This is why classical simulation is hard:")
print("  50 qubits → 2⁵⁰ × 2⁵⁰ = 10¹⁵ × 10¹⁵ matrix")
print("  That's more numbers than atoms in a grain of sand.")
print()
print("Tensor product rules (Qiskit ordering: qubit 0 = rightmost):")
print("  • Gate on qubit 1 only: U ⊗ I")
print("  • Gate on qubit 0 only: I ⊗ U")

In [ ]:
# ── Build multi-qubit gate matrices with np.kron ──
print("🔧 Building Gate Matrices with np.kron")
print("=" * 55)
print()

# Example: H on qubit 1, nothing on qubit 0
H_on_q1 = np.kron(H_mat, I2)
print("H ⊗ I (H on qubit 1):")
print(H_on_q1.round(4))
print()

# Verify against Qiskit
qc = QuantumCircuit(2); qc.h(1)
qiskit_H_q1 = Operator(qc).data
print(f"Matches Qiskit: {np.allclose(H_on_q1, qiskit_H_q1)} ✅")
print()

# Example: X on qubit 0, nothing on qubit 1
X_on_q0 = np.kron(I2, X)
print("I ⊗ X (X on qubit 0):")
print(X_on_q0.real.astype(int))
print()

qc = QuantumCircuit(2); qc.x(0)
print(f"Matches Qiskit: {np.allclose(X_on_q0, Operator(qc).data)} ✅")
print()

# Full circuit as matrix multiplication
print("Circuit = Sequential matrix multiplication:")
print("  If circuit does: H on q1, then X on q0")
print("  Matrix = (I⊗X) · (H⊗I)  ← rightmost gate first!")
circuit_matrix = X_on_q0 @ H_on_q1
qc = QuantumCircuit(2); qc.h(1); qc.x(0)
print(f"  Matches Qiskit: {np.allclose(circuit_matrix, Operator(qc).data)} ✅")

---
## Section 2: The CNOT Matrix — Projector Construction

The CNOT matrix uses **projectors** — the most general way to build controlled gates:

$$\text{CNOT} = \vert 0\rangle\langle 0\vert \otimes I + \vert 1\rangle\langle 1\vert \otimes X$$

Read this as: "If control is $\vert 0\rangle$, do nothing. If control is $\vert 1\rangle$, apply X."

Where:
- $\vert 0\rangle\langle 0\vert = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}$ — projector onto $\vert 0\rangle$
- $\vert 1\rangle\langle 1\vert = \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix}$ — projector onto $\vert 1\rangle$

In [ ]:
print("🔗 CNOT from Projectors")
print("=" * 55)
print()

# Build CNOT: |0⟩⟨0| ⊗ I + |1⟩⟨1| ⊗ X
CNOT = np.kron(P0, I2) + np.kron(P1, X)
print("CNOT = |0⟩⟨0| ⊗ I + |1⟩⟨1| ⊗ X =")
print(CNOT.real.astype(int))
print()

# Verify
qc = QuantumCircuit(2); qc.cx(1, 0)
print(f"Matches Qiskit cx(1,0): {np.allclose(CNOT, Operator(qc).data)} ✅")
print()

# Walk through each basis state
print("CNOT truth table (algebraic verification):")
basis_states = {
    '|00⟩': np.array([1, 0, 0, 0]),
    '|01⟩': np.array([0, 1, 0, 0]),
    '|10⟩': np.array([0, 0, 1, 0]),
    '|11⟩': np.array([0, 0, 0, 1]),
}
labels = ['|00⟩', '|01⟩', '|10⟩', '|11⟩']
for name, state in basis_states.items():
    result = CNOT @ state
    result_name = labels[np.argmax(np.abs(result))]
    print(f"  CNOT{name} = {result.real.astype(int)} = {result_name}")

print()
print("Key properties of CNOT:")
print(f"  Self-inverse (CNOT² = I): {np.allclose(CNOT @ CNOT, np.eye(4))} ✅")
print(f"  Unitary: {np.allclose(CNOT @ CNOT.conj().T, np.eye(4))} ✅")

---
## Section 3: SWAP Gate and Its CNOT Decomposition

**SWAP** exchanges two qubits: $\vert ab\rangle \to \vert ba\rangle$

$$\text{SWAP} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \end{pmatrix}$$

SWAP is **not a native gate** on most hardware. It must be decomposed:

$$\text{SWAP} = \text{CNOT}(0{\to}1) \cdot \text{CNOT}(1{\to}0) \cdot \text{CNOT}(0{\to}1)$$

**Cost:** 3 CNOTs → this is why SWAP routing is so expensive!

In [ ]:
print("🔄 SWAP Gate and Its CNOT Decomposition")
print("=" * 55)
print()

# Build SWAP matrix
SWAP = np.array([
    [1, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1]
], dtype=complex)

print("SWAP matrix:")
print(SWAP.real.astype(int))
print()

# Verify it swaps
print("SWAP truth table:")
for name, state in basis_states.items():
    result = SWAP @ state
    result_name = labels[np.argmax(np.abs(result))]
    print(f"  SWAP{name} = {result_name}")

print()
print(f"SWAP is self-inverse (SWAP² = I): {np.allclose(SWAP @ SWAP, np.eye(4))} ✅")
print()

# ── Decompose SWAP into 3 CNOTs ──
print("SWAP = CNOT(0→1) · CNOT(1→0) · CNOT(0→1)")
print()

# Build the three CNOTs
CNOT_01 = np.kron(I2, P0) + np.kron(X, P1)  # control=0, target=1
CNOT_10 = np.kron(P0, I2) + np.kron(P1, X)  # control=1, target=0

SWAP_from_cnots = CNOT_01 @ CNOT_10 @ CNOT_01
print(f"3 CNOTs match SWAP: {np.allclose(SWAP_from_cnots, SWAP)} ✅")
print()

# Qiskit circuit
qc_swap = QuantumCircuit(2)
qc_swap.cx(0, 1)
qc_swap.cx(1, 0)
qc_swap.cx(0, 1)
print("Circuit:")
print(qc_swap.draw())
print()
print(f"Qiskit verification: {np.allclose(Operator(qc_swap).data, SWAP)} ✅")
print()
print("💡 SWAP costs 3 CNOTs. On hardware where CNOT takes ~300ns and")
print("   has ~1% error, each SWAP adds ~900ns and ~3% error.")
print("   Minimizing SWAP count is THE key transpiler optimization.")

---
## Section 4: Controlled Gates — The General Recipe

Any single-qubit gate $U$ can be made **conditional**:

$$\text{C-}U = \vert 0\rangle\langle 0\vert \otimes I + \vert 1\rangle\langle 1\vert \otimes U$$

| Controlled Gate | U | Matrix Form |
| :--- | :--- | :--- |
| CNOT (CX) | X | diag + anti-diag |
| CZ | Z | diag(1, 1, 1, −1) |
| CS | S | diag(1, 1, 1, i) |
| CT | T | diag(1, 1, 1, $e^{i\pi/4}$) |

### CZ Is Special
CZ = diag(1, 1, 1, −1) is **symmetric**: swapping control and target gives the same gate. This is why CZ is the native entangling gate on many platforms.

In [ ]:
print("🎛️ Controlled Gates")
print("=" * 55)
print()

def controlled_U(U, name="U"):
    """Build and display a controlled-U gate."""
    CU = np.kron(P0, I2) + np.kron(P1, U)
    return CU

# Build several controlled gates
gates = {
    'CX (CNOT)': (X, 'cx'),
    'CZ': (Z, 'cz'),
    'CS': (S, None),
    'CT': (T, None),
}

for name, (U, qiskit_name) in gates.items():
    CU = controlled_U(U, name)
    print(f"{name}:")
    # Show diagonal for diagonal gates, full matrix otherwise
    diag = np.diag(CU)
    if np.allclose(CU, np.diag(diag)):
        print(f"  diag = [{', '.join(f'{d:.4f}' for d in diag)}]")
    else:
        print(f"  matrix:")
        for row in CU:
            print(f"    [{', '.join(f'{v.real:+.2f}{v.imag:+.2f}j' if v.imag else f'{v.real:+.2f}' for v in row)}]")

    if qiskit_name:
        qc = QuantumCircuit(2)
        getattr(qc, qiskit_name)(1, 0)
        match = np.allclose(CU, Operator(qc).data)
        print(f"  Matches Qiskit: {match} ✅")
    print()

# Show CZ symmetry
CZ = controlled_U(Z)
print("CZ symmetry (control ↔ target):")
CZ_swapped = np.kron(I2, P0) + np.kron(Z, P1)  # swap roles
print(f"  CZ(1→0) = CZ(0→1): {np.allclose(CZ, CZ_swapped)} ✅")
print("  → CZ doesn't care which qubit is the control!")
print()

# CZ from CNOT
print("CZ = H · CNOT · H (on target):")
qc_cz = QuantumCircuit(2)
qc_cz.h(0)
qc_cz.cx(1, 0)
qc_cz.h(0)
print(f"  Matches CZ: {np.allclose(Operator(qc_cz).data, CZ)} ✅")
print()
print("💡 This is a general pattern: to change a controlled gate,")
print("   conjugate the target with basis-change unitaries.")
print("   CZ = (I⊗H) · CNOT · (I⊗H) because HXH = Z.")

---
## Section 5: The Toffoli Gate (CCX)

The **Toffoli gate** is a doubly-controlled NOT:
- 3 qubits: 2 controls + 1 target
- Flips target only when **both** controls are $\vert 1\rangle$
- It's a **universal classical gate** (can implement AND, OR, NOT)
- 8×8 matrix: identity everywhere except $\vert 110\rangle \leftrightarrow \vert 111\rangle$

### Decomposition Cost
Toffoli requires at minimum **6 CNOTs** to decompose into 1- and 2-qubit gates.  
The standard decomposition also uses **T and T† gates** — making it non-Clifford.

In [ ]:
print("⚡ The Toffoli Gate (CCX)")
print("=" * 55)
print()

# Build Toffoli matrix
toffoli = np.eye(8, dtype=complex)
toffoli[6, 6] = 0; toffoli[7, 7] = 0
toffoli[6, 7] = 1; toffoli[7, 6] = 1

print("Toffoli matrix (8×8):")
print(toffoli.real.astype(int))
print()

# Verify with Qiskit
qc_tof = QuantumCircuit(3); qc_tof.ccx(2, 1, 0)
print(f"Matches Qiskit ccx(2,1,0): {np.allclose(toffoli, Operator(qc_tof).data)} ✅")
print()

# Truth table
print("Toffoli truth table:")
print(f"  {'Input':<8s} {'Output':<8s} {'Action'}")
print("-" * 40)
for i in range(8):
    state = np.zeros(8); state[i] = 1
    result = toffoli @ state
    j = np.argmax(np.abs(result))
    in_bits = f"|{i:03b}⟩"
    out_bits = f"|{j:03b}⟩"
    action = "FLIP" if i != j else "—"
    print(f"  {in_bits:<8s} {out_bits:<8s} {action}")

print()
print("💡 Only |110⟩↔|111⟩ are swapped: both controls must be |1⟩.")

In [ ]:
# ── Toffoli decomposition ──
print("Toffoli Decomposition into 1- and 2-qubit gates")
print("=" * 55)
print()

qc_tof = QuantumCircuit(3)
qc_tof.ccx(2, 1, 0)

# Decompose once
decomposed = qc_tof.decompose()
print("Decomposed circuit:")
print(decomposed.draw(fold=120))
print()

# Count gates
gate_counts = {}
for inst in decomposed.data:
    name = inst.operation.name
    gate_counts[name] = gate_counts.get(name, 0) + 1

print("Gate counts after decomposition:")
for gate, count in sorted(gate_counts.items()):
    print(f"  {gate}: {count}")

total_cnots = gate_counts.get('cx', 0)
print(f"\nTotal CNOTs: {total_cnots}")
print(f"Matches Toffoli: {np.allclose(Operator(decomposed).data, toffoli)} ✅")
print()
print("💡 6 CNOTs is the minimum for Toffoli decomposition.")
print("   Each CNOT takes ~300ns on hardware → Toffoli ≈ 2μs total.")
print("   The T gates make it non-Clifford → can't be classically simulated.")

---
## Section 6: Circuit Identities

Circuit identities are the "algebra" of quantum computing. Knowing them lets you optimize circuits and understand transpiler behavior.

| Identity | Why It Matters |
| :--- | :--- |
| $HXH = Z$ | Conjugation swaps X and Z bases |
| $HZH = X$ | Same identity, other direction |
| $HYH = -Y$ | Y picks up a sign |
| $SXS^\dagger = Y$ | S rotates X into Y in the Bloch sphere |
| $T^2 = S$ | T is the "square root" of S |
| $S^2 = Z$ | S is the "square root" of Z |
| $\text{CNOT}^2 = I$ | CNOT is self-inverse |
| 3 CNOTs = SWAP | Fundamental routing cost |

In [ ]:
print("🔀 Circuit Identities")
print("=" * 55)
print()

identities = [
    ("HXH = Z",      H_mat @ X @ H_mat, Z),
    ("HZH = X",      H_mat @ Z @ H_mat, X),
    ("HYH = -Y",     H_mat @ Y @ H_mat, -Y),
    ("SXS† = Y",     S @ X @ S.conj().T, Y),
    ("T² = S",       T @ T, S),
    ("S² = Z",       S @ S, Z),
    ("X² = I",       X @ X, I2),
    ("Z² = I",       Z @ Z, I2),
    ("H² = I",       H_mat @ H_mat, I2),
    ("(XY)³ = -I",   np.linalg.matrix_power(X @ Y, 3), -I2),
]

for name, computed, expected in identities:
    match = np.allclose(computed, expected)
    status = "✅" if match else "❌"
    print(f"  {status} {name}")

print()

# Multi-qubit identities
print("Multi-qubit identities:")
print(f"  ✅ CNOT² = I: {np.allclose(CNOT @ CNOT, np.eye(4))}")
print(f"  ✅ SWAP² = I: {np.allclose(SWAP @ SWAP, np.eye(4))}")

# CZ = (I⊗H) · CNOT · (I⊗H)
CZ = np.kron(P0, I2) + np.kron(P1, Z)
CZ_from_cnot = np.kron(I2, H_mat) @ CNOT @ np.kron(I2, H_mat)
print(f"  ✅ CZ = (I⊗H)·CNOT·(I⊗H): {np.allclose(CZ_from_cnot, CZ)}")

print()
print("💡 These identities are your optimization toolkit.")
print("   Transpilers use them automatically — but knowing them")
print("   helps you write better circuits and debug transpiler output.")

---
## Section 7: Universal Gate Sets

A **universal gate set** can approximate **any** $n$-qubit unitary to arbitrary precision.

### The Standard Universal Set: {H, T, CNOT}

| Component | Role | Why |
| :--- | :--- | :--- |
| H (Hadamard) | Basis change | Creates superposition, swaps X↔Z |
| T ($\pi/8$ gate) | Phase precision | Non-Clifford, enables fine rotations |
| CNOT | Entanglement | Couples qubits, extends to multi-qubit ops |

**Why these three?**
1. **H and T** together generate a **dense subset** of all single-qubit unitaries (SU(2))
2. **CNOT** extends single-qubit universality to multi-qubit universality
3. The **Solovay-Kitaev theorem** guarantees efficiency: any gate can be approximated to precision $\epsilon$ using $O(\log^c(1/\epsilon))$ gates

### The Gate Hierarchy

$$T \xrightarrow{T^2} S \xrightarrow{S^2} Z \xrightarrow{HZH} X \xrightarrow{SXS^\dagger} Y$$

From just H and T, you can build ALL Pauli and Clifford gates!

In [ ]:
print("🌐 Universal Gate Sets: {H, T, CNOT}")
print("=" * 55)
print()

# Show the gate hierarchy
print("Building ALL standard gates from {H, T}:")
print()

S_from_T = T @ T
Z_from_S = S_from_T @ S_from_T
X_from_HZH = H_mat @ Z_from_S @ H_mat
Y_from_SXSd = S_from_T @ X_from_HZH @ S_from_T.conj().T

print(f"  T² = S:    {np.allclose(S_from_T, S)} ✅")
print(f"  S² = Z:    {np.allclose(Z_from_S, Z)} ✅")
print(f"  HZH = X:   {np.allclose(X_from_HZH, X)} ✅")
print(f"  SXS† = Y:  {np.allclose(Y_from_SXSd, Y)} ✅")
print()

# Show that H and T generate irrational rotations
print("Why T provides universality:")
print(f"  T = Rz(π/4): rotation by π/4 around Z")
print(f"  H = rotation that swaps X and Z axes")
print(f"  HT = rotation by an IRRATIONAL angle on the Bloch sphere")
print()

# Demonstrate HT generates dense set
HT = H_mat @ T
print("Successive applications of HT:")
print("  Powers of (HT) generate increasingly fine rotations.")
print("  Because the angle is irrational, they NEVER repeat —")
print("  they densely fill SU(2), like irrational rotations on a circle.")
print()

# Show some powers
gate = np.eye(2, dtype=complex)
angles = []
for k in range(1, 25):
    gate = HT @ gate
    # Extract rotation angle from trace: Tr(U) = 2cos(θ/2)
    cos_half = np.real(np.trace(gate)) / 2
    cos_half = np.clip(cos_half, -1, 1)
    angle = 2 * np.arccos(abs(cos_half))
    angles.append(angle * 180 / np.pi)

print(f"  Rotation angles (degrees) for (HT)^k, k=1..24:")
for i in range(0, 24, 6):
    chunk = [f"{a:6.1f}°" for a in angles[i:i+6]]
    print(f"    k={i+1:2d}-{i+6:2d}: {', '.join(chunk)}")

# Verify no repetitions
unique_angles = len(set(round(a, 3) for a in angles))
print(f"\n  Unique angles in 24 powers: {unique_angles} (all different!)")
print()
print("💡 This is the Solovay-Kitaev idea: because HT generates")
print("   irrational rotations, repeated application covers ALL of SU(2).")
print("   Any rotation can be approximated to precision ε with")
print("   O(log^c(1/ε)) gates. That's EFFICIENT approximation.")

---
## Section 8: Circuit Simulation with Qiskit Aer

So far we've used `Statevector` — exact simulation that gives the full quantum state.  
Real quantum computers don't give you the statevector. They give you **samples**.

**Qiskit Aer** simulates this sampling process:
- Add **measurements** to your circuit
- Run for a number of **shots** (repetitions)
- Get back **counts** — how many times each outcome occurred

This is closer to what real hardware gives you.

In [ ]:
print("🎲 Circuit Simulation with Qiskit Aer")
print("=" * 55)
print()

# ── Bell state with measurements ──
qc_bell = QuantumCircuit(2, 2)  # 2 qubits, 2 classical bits
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure([0, 1], [0, 1])

print("Bell state circuit with measurements:")
print(qc_bell.draw())
print()

# Run simulation
sim = AerSimulator()
shots = 4096
result = sim.run(qc_bell, shots=shots).result()
counts = result.get_counts()

print(f"Results ({shots} shots):")
for outcome, count in sorted(counts.items()):
    pct = count / shots * 100
    bar = '█' * int(pct / 2)
    print(f"  |{outcome}⟩: {count:>5d} ({pct:5.1f}%) {bar}")

print()
print("Expected: ~50% |00⟩ and ~50% |11⟩ (never |01⟩ or |10⟩)")
print()

# ── GHZ state ──
print("4-qubit GHZ state:")
n = 4
qc_ghz = QuantumCircuit(n, n)
qc_ghz.h(0)
for i in range(n - 1):
    qc_ghz.cx(i, i + 1)
qc_ghz.measure(range(n), range(n))

result = sim.run(qc_ghz, shots=shots).result()
counts = result.get_counts()

print(f"Results ({shots} shots):")
for outcome, count in sorted(counts.items()):
    pct = count / shots * 100
    bar = '█' * int(pct / 2)
    print(f"  |{outcome}⟩: {count:>5d} ({pct:5.1f}%) {bar}")

print()
print("Expected: ~50% |0000⟩ and ~50% |1111⟩")

In [ ]:
# ── Visualize shot-based statistics ──
print("📊 Shot-Based Statistics Visualization")
print("=" * 55)
print()

# Run at different shot counts
shot_counts = [10, 100, 1000, 10000]

fig = make_subplots(
    rows=1, cols=len(shot_counts),
    subplot_titles=[f'{s} shots' for s in shot_counts]
)

for i, shots in enumerate(shot_counts, 1):
    result = sim.run(qc_bell, shots=shots).result()
    counts = result.get_counts()

    outcomes = ['00', '01', '10', '11']
    probs = [counts.get(o, 0) / shots for o in outcomes]

    colors = ['#2ecc71' if o in ['00', '11'] else '#e74c3c' for o in outcomes]

    fig.add_trace(go.Bar(
        x=outcomes, y=probs, name=f'{shots}',
        marker_color=colors,
        text=[f'{p:.2f}' for p in probs],
        textposition='auto',
        showlegend=False
    ), row=1, col=i)

    # Add ideal line
    fig.add_hline(y=0.5, line_dash='dash', line_color='gray',
                  row=1, col=i)

fig.update_layout(
    title='Bell State |Φ+⟩: Convergence to Ideal Probabilities',
    width=1100, height=400,
    yaxis_title='Probability'
)
fig.update_yaxes(range=[0, 0.7])
fig.show()

print()
print("💡 With few shots, results are noisy. With many shots, they")
print("   converge to the ideal probabilities (50/50 for Bell state).")
print("   Real quantum computers add HARDWARE noise on top of this")
print("   statistical noise — that's tomorrow's topic (Day 5).")

---
## 🎯 Day 4 Summary

### What you learned today:
1. **Circuit-Matrix Correspondence** — Every circuit IS a unitary matrix; build them with `np.kron`
2. **CNOT from Projectors** — $\vert 0\rangle\langle 0\vert \otimes I + \vert 1\rangle\langle 1\vert \otimes X$
3. **SWAP = 3 CNOTs** — SWAP routing is the transpiler's biggest expense
4. **Controlled-U Recipe** — $\vert 0\rangle\langle 0\vert \otimes I + \vert 1\rangle\langle 1\vert \otimes U$ for any U
5. **CZ Symmetry** — CZ doesn't care which qubit is the control
6. **Toffoli = 6 CNOTs** — Universal classical gate, expensive on quantum hardware
7. **Circuit Identities** — $HXH=Z$, $T^2=S$, $S^2=Z$, CZ from H+CNOT+H
8. **{H, T, CNOT} is Universal** — Solovay-Kitaev guarantees efficient approximation
9. **Aer Simulator** — Shot-based sampling like real hardware

### Key takeaway for interviews:
> "A gate set is universal if it can approximate any unitary to arbitrary precision. {H, T, CNOT} achieves this because H and T generate irrational rotations that densely fill SU(2), and CNOT extends this to multi-qubit operations. In the fault-tolerant era, Clifford gates are essentially free but each T gate costs ~1000 physical qubits for magic state distillation — so T-count optimization is the defining challenge."

---
**Tomorrow (Day 5):** Noise & Reality — density matrices, quantum channels, depolarizing/amplitude damping noise, Qiskit Aer noise models, and why NISQ is hard.